In [91]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [92]:
training_data = datasets.MNIST(
    "data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.MNIST(
    "data",
    train=False,
    download=True,
    transform=ToTensor()
)
# print(test_data)

In [93]:
BATCH_SIZE = 128

train_dataloader = DataLoader(
    training_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_dataloader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE
)

for x, y in train_dataloader:
    print(f"shape of x[N, C, W, H]:{x.shape}")
    print(f"share of y: {y.shape}, {y.dtype}")
    print(f"lable of y: {y[0].item()}")
    break

shape of x[N, C, W, H]:torch.Size([128, 1, 28, 28])
share of y: torch.Size([128]), torch.int64
lable of y: 2


In [94]:
from torch.cuda import is_available
from torch.nn.modules.linear import Linear
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.nmodel = nn.Sequential(
            nn.Linear(28*28, 256),
            nn.BatchNorm1d(256), # 后加优化用
            nn.ReLU(),
            nn.Dropout(0.5), # 后加优化用
            nn.Linear(256, 128),
            nn.BatchNorm1d(128), # 后加优化用
            nn.ReLU(),
            nn.Dropout(0.2), # 后加优化用
            nn.Linear(128, 10)
        )
    def forward(self, x):
      x = nn.Flatten()(x)
      output = self.nmodel(x)
      return output

device = ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"GPU->{device}")

model = NeuralNetwork().to(device)

GPU->cuda


In [95]:
loss_fn = nn.CrossEntropyLoss()
learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(x)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:5d}]")



In [96]:
def test(dataloader, model, loss_fn):
    num_batches = len(dataloader)
    size = len(dataloader.dataset)
    test_loss, correct = 0, 0
    model.eval()
    with torch.no_grad():
        for batch, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            pred = model(x)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        test_loss /= num_batches
        correct /= size

    print(f"test Error: \n Accuracy: {(100*correct):>0.1f}%, avg loss: {test_loss: > 8f}")



In [98]:
epochs = 20

for epoch in range(epochs):
    print(f"{epoch+1}/{epochs}")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)

print("Finished traning!")

1/20
loss: 0.105309 [    0/60000]
loss: 0.087690 [12800/60000]
loss: 0.111991 [25600/60000]
loss: 0.078638 [38400/60000]
loss: 0.038059 [51200/60000]
test Error: 
 Accuracy: 98.2%, avg loss:  0.058190
2/20
loss: 0.079773 [    0/60000]
loss: 0.126437 [12800/60000]
loss: 0.109726 [25600/60000]
loss: 0.059336 [38400/60000]
loss: 0.146049 [51200/60000]
test Error: 
 Accuracy: 98.1%, avg loss:  0.060470
3/20
loss: 0.162798 [    0/60000]
loss: 0.061943 [12800/60000]
loss: 0.055590 [25600/60000]
loss: 0.063264 [38400/60000]
loss: 0.046486 [51200/60000]
test Error: 
 Accuracy: 98.3%, avg loss:  0.055460
4/20
loss: 0.104248 [    0/60000]
loss: 0.042423 [12800/60000]
loss: 0.033455 [25600/60000]
loss: 0.102902 [38400/60000]
loss: 0.054273 [51200/60000]
test Error: 
 Accuracy: 98.1%, avg loss:  0.059431
5/20
loss: 0.027279 [    0/60000]
loss: 0.052341 [12800/60000]
loss: 0.024174 [25600/60000]
loss: 0.065699 [38400/60000]
loss: 0.044771 [51200/60000]
test Error: 
 Accuracy: 98.3%, avg loss:  0.05

In [99]:
torch.save(model.state_dict(), "model_weights.pth")
print("saved pytorch model")

saved pytorch model
